# PCA — Principal Component Analysis
From scratch — interview-ready

## What is PCA?

PCA finds the directions in your data where **variance is highest** and projects the data onto those directions.

**Why?**
```
100 features  →  many are correlated or useless
PCA           →  find the 2-3 directions that capture most of the information
result        →  compress 100 features → 2 features, losing as little info as possible
```

**Intuition:**
```
imagine a cloud of points spread diagonally in 2D

      *  *
   *  *  *
*  *  *

PC1 = the long diagonal axis  ← most variance here
PC2 = perpendicular to PC1    ← remaining variance
```
PCA rotates your coordinate system to align with those axes.

## Step-by-Step: How PCA Works

```
Input: X of shape (m, n)   m = samples, n = features
```

---

**Step 1 — Center the data**
```
X = X - mean(X, axis=0)

why?  PCA measures variance (spread), not position.
      centering removes the mean so variance is measured from zero.
      if you skip this, the first component just points toward the mean.
```

---

**Step 2 — Covariance matrix**
```
cov = (1/m) × XᵀX       shape: (n, n)

why?  covariance matrix tells you how each pair of features varies together.
      cov[i,j] = high  →  feature i and j move together
      cov[i,j] = 0     →  features are independent
      diagonal         →  variance of each feature
```

---

**Step 3 — Eigen decomposition**
```
eigenvalues, eigenvectors = eig(cov)

eigenvector  →  a direction (principal component)
eigenvalue   →  how much variance that direction captures

cov @ v = λ × v
  v = direction that only stretches, never rotates
  λ = how much stretch
```

---

**Step 4 — Sort by eigenvalue (largest first)**
```
largest eigenvalue = direction of most variance = PC1
second largest     = PC2
...

take top k eigenvectors → shape (n, k)
```

---

**Step 5 — Project**
```
X_reduced = X @ eigenvectors[:, :k]     shape: (m, k)

this is change of basis — express data in the new coordinate system
each row = same data point, now described using k principal components
```

## Small Example (By Hand)

```
X = [[2, 3],
     [4, 5],
     [6, 7]]
```

**Step 1 — Center:**
```
mean = [4, 5]
X_centered = [[-2, -2],
               [ 0,  0],
               [ 2,  2]]
```

**Step 2 — Covariance:**
```
cov = [[4, 4],
       [4, 4]]

features perfectly correlated (both move together)
```

**Step 3 — Eigenvalues:**
```
λ1 = 8,  v1 = [0.707, 0.707]   ← diagonal direction, captures all variance
λ2 = 0,  v2 = [-0.707, 0.707]  ← perpendicular, zero variance
```

**Step 4 — Take top 1 component (k=1):**
```
v1 = [0.707, 0.707]
```

**Step 5 — Project:**
```
X_reduced = X_centered @ v1
          = [[-2.83],
             [ 0.00],
             [ 2.83]]

2 features → 1 feature, zero information lost (λ2 = 0 anyway)
```

## PCA — Method 1: Eigen Decomposition

In [ ]:
import numpy as np

def pca_eig(X, n_components):
    X = np.array(X, dtype=float)

    # step 1: center
    X = X - X.mean(axis=0)

    # step 2: covariance matrix
    cov = np.cov(X.T)                          # shape (n, n)

    # step 3: eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eig(cov)

    # step 4: sort by largest eigenvalue
    idx = np.argsort(eigenvalues)[::-1]
    eigenvectors = eigenvectors[:, idx]        # columns = eigenvectors

    # step 5: project onto top k components
    components = eigenvectors[:, :n_components]
    return X @ components


X = [[2.5, 2.4],
     [0.5, 0.7],
     [2.2, 2.9],
     [1.9, 2.2],
     [3.1, 3.0]]

result = pca_eig(X, n_components=1)
print("Shape:", result.shape)
print(result)

## PCA — Method 2: SVD (more numerically stable)

```
SVD decomposes A = U × S × Vᵀ

U    →  left singular vectors   (m, m)
S    →  singular values         (k,)      ← square roots of eigenvalues of XᵀX
Vt   →  right singular vectors  (n, n)    ← same as eigenvectors of covariance matrix

so instead of computing covariance + eig,
just run SVD on X directly → Vt rows = principal components
```

In [ ]:
import numpy as np

def pca_svd(X, n_components):
    X = np.array(X, dtype=float)

    # step 1: center
    X = X - X.mean(axis=0)

    # step 2: SVD
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    # Vt rows = principal components (already sorted by variance)

    # step 3: project
    components = Vt[:n_components].T          # shape (n, k)
    return X @ components


X = [[2.5, 2.4],
     [0.5, 0.7],
     [2.2, 2.9],
     [1.9, 2.2],
     [3.1, 3.0]]

result = pca_svd(X, n_components=1)
print("Shape:", result.shape)
print(result)

## Explained Variance — How Many Components to Keep?

```
explained_variance_ratio = eigenvalue_i / sum(all eigenvalues)

tells you: what % of total variance does PC_i capture?

common rule: keep enough components to explain 95% variance
```

In [ ]:
import numpy as np

def explained_variance(X):
    X = np.array(X, dtype=float)
    X = X - X.mean(axis=0)

    cov = np.cov(X.T)
    eigenvalues, _ = np.linalg.eig(cov)
    eigenvalues = np.sort(eigenvalues)[::-1]

    ratios = eigenvalues / eigenvalues.sum()
    cumulative = np.cumsum(ratios)

    for i, (r, c) in enumerate(zip(ratios, cumulative)):
        print(f"PC{i+1}: {r:.2%} variance  |  cumulative: {c:.2%}")


X = [[2.5, 2.4],
     [0.5, 0.7],
     [2.2, 2.9],
     [1.9, 2.2],
     [3.1, 3.0]]

explained_variance(X)

## Interview Questions on PCA

**Q: Why do we center the data before PCA?**
```
PCA finds directions of max variance.
Without centering, the first PC points toward the mean, not max variance.
Centering removes the mean so variance is measured from zero.
```

**Q: Why use SVD instead of eigen decomposition?**
```
Computing XᵀX amplifies numerical errors for large matrices.
SVD works directly on X — more stable for real/noisy data.
sklearn uses SVD internally.
```

**Q: What does each eigenvalue represent?**
```
eigenvalue = amount of variance captured by that principal component
large eigenvalue → important direction
near-zero eigenvalue → redundant direction, safe to drop
```

**Q: Can PCA be used for non-linear data?**
```
No — PCA only finds linear directions.
For non-linear: use Kernel PCA or t-SNE or UMAP.
```

**Q: What is the relationship between PCA and SVD?**
```
A = U S Vᵀ

singular values S   = sqrt(eigenvalues of XᵀX)
right vectors Vt    = eigenvectors of covariance matrix = principal components

SVD on X  ==  eigen decomposition on covariance matrix
```